# Gun Violence Twitter Data Pipeline

This notebook consolidates the entire data processing pipeline: configuration, bounding box collection, filtering by locations/dates, and keyword filtering.

## 1. Configuration & Setup

In [1]:
import os
import re
import json
import gzip
import glob
import pandas as pd
from multiprocessing import Pool
from datetime import timedelta
from tqdm import tqdm
from geopy.geocoders import Nominatim
import time
import requests
import math

print("=== Initializing Pipeline Configuration ===")

# ==========================================
# FILE PATHS
# ==========================================
RAW_DATA_DIR = "/n/holylabs/LABS/cga/Lab/data/geo-tweets/cga-sbg-tweets/"
OUTPUT_DIR = "/n/home10/jrochatrindade/gun-violence-twitter-forecasting/data/processed/"
BBOX_FILE = "/n/home10/jrochatrindade/gun-violence-twitter-forecasting/data/bboxes.json"

print(f" Input Directory: {RAW_DATA_DIR}")
print(f" Output Directory: {OUTPUT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("  -> Verified/Created Output Directory.")

# ==========================================
# INCIDENTS & DATES
# ==========================================
TARGET_YEARS = ["2022", "2023"]
INCIDENTS = {
    "Uvalde, TX": "2022-05-24",
    "Highland Park, IL": "2022-07-04",
    "Monterey Park, CA": "2023-01-21",
    "Nashville, TN": "2023-03-27"
}

print(f" Target Years scoped: {TARGET_YEARS}")
print(f" Target Incidents loaded: {list(INCIDENTS.keys())}")

# ==========================================
# KEYWORDS & HASHTAGS
# ==========================================
HASHTAGS = [
    "#GunViolence", "#GunControl", "#GunReformNow", "#BanAssaultWeaponsNow", "#Parkland",
    "#SecondAmendment", "#2A", "#2ndAmendment", "#gunsafety", "#GunControlNow", "#ConcealedCarry",
    "#GunRights", "#NeverAgain", "#NRATerrorism", "#ProGun", "#NeverAlone", "#MarchForOurLives",
    "#OpenCarry", "#gunowners", "#GunDebate", "#FloridaShooting", "#massshooting",
    "#MentalHealthAwareness", "#SuicidePrevention", "#EndGunViolence", "#GunSense",
    "#GunSafetyNow", "#DisarmHate", "#StopGunViolence", "#GunViolencePrevention",
    "#SaveOurKids", "#EnoughIsEnough", "#StopTheViolence"
]

KEYWORDS = [
    "gun violence", "mass shooting", "gun control", "gun deaths", "firearm fatalities",
    "shooting incident", "accidental shooting", "gun accident", "firearm ownership",
    "assault weapons", "gun policy", "active shooter", "gun rally", "shotgun", "rifle",
    "handgun", "automatic weapons", "concealed weapon", "mental health crisis",
    "suicide by firearm", "accidental death", "gun reform", "firearm injuries",
    "gun ban", "shooting prevention", "gun legislation", "open carry", "self-defense shooting",
    "background checks", "anger", "fear", "pain", "grief", "advocacy", "hate", "sadness", 
    "rage", "blame", "cause", "responsibility", "Parkland shooting", "Sandy Hook", 
    "Las Vegas shooting", "Orlando nightclub shooting", "school shooting", "massacre",
    "income inequality", "education level", "poverty", "unemployment", "community violence",
    "neighborhood safety", "urban violence", "race and gun violence", "social isolation",
    "gang violence", "threat", "warning signs", "premeditated", "manifesto", "violent intentions",
    "planned attack", "threat assessment", "legislation", "gun registration", 
    "universal background checks", "red flag laws", "gun safety legislation", "gun lobby", 
    "firearm restriction", "buyback program", "law enforcement", "community policing", 
    "violence prevention", "intervention", "risk assessment", "emergency response", 
    "public safety", "crisis intervention", "strap", "heater", "gat", "piece", "pop off", 
    "clap back", "drill", "spin the block", "active shooter drill", "lockdown", "gun show", 
    "ammo shortage", "arms dealer", "illegal firearms", "3D-printed gun", "ghost gun"
]

print(f" Processing {len(HASHTAGS)} hashtags and {len(KEYWORDS)} keywords...")
hashtag_words = [h.replace("#", "").lower() for h in HASHTAGS]
all_phrases = list(set([k.lower() for k in KEYWORDS] + hashtag_words))

escaped_phrases = [re.escape(phrase) for phrase in all_phrases]
REGEX_PATTERN = re.compile(r"\b(?:" + "|".join(escaped_phrases) + r")\b", flags=re.IGNORECASE)
print(f"  -> Compiled search vocabulary: {len(all_phrases)} unique token groups generated.")
print("=== Setup Successful. Ready for Spatio-Temporal Extraction ===")

=== Initializing Pipeline Configuration ===
 Input Directory: /n/holylabs/LABS/cga/Lab/data/geo-tweets/cga-sbg-tweets/
 Output Directory: /n/home10/jrochatrindade/gun-violence-twitter-forecasting/data/processed/
  -> Verified/Created Output Directory.
 Target Years scoped: ['2022', '2023']
 Target Incidents loaded: ['Uvalde, TX', 'Highland Park, IL', 'Monterey Park, CA', 'Nashville, TN']
 Processing 33 hashtags and 95 keywords...
  -> Compiled search vocabulary: 128 unique token groups generated.
=== Setup Successful. Ready for Spatio-Temporal Extraction ===


## 2. Collect Bounding Boxes

In [6]:
def collect_bounding_boxes(locations, output_file=BBOX_FILE, buffer_km=0, email=None):
    bboxes = {}
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    def _coords_from_geojson(geo):
        pts = []
        t = geo.get('type')
        if t == 'Polygon':
            for ring in geo.get('coordinates', []):
                for lon, lat in ring:
                    pts.append((lat, lon))
        elif t == 'MultiPolygon':
            for poly in geo.get('coordinates', []):
                for ring in poly:
                    for lon, lat in ring:
                        pts.append((lat, lon))
        elif t == 'Point':
            lon, lat = geo.get('coordinates', [None, None])
            if lon is not None:
                pts.append((lat, lon))
        elif t == 'GeometryCollection':
            for g in geo.get('geometries', []):
                pts.extend(_coords_from_geojson(g))
        else:
            def walk(c):
                if isinstance(c, list) and len(c) >= 2 and isinstance(c[0], (int, float)):
                    lon, lat = c[0], c[1]
                    pts.append((lat, lon))
                elif isinstance(c, list):
                    for item in c:
                        walk(item)
            walk(geo.get('coordinates'))
        return pts

    NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
    bboxes = {}
    headers = {"User-Agent": f"gun_violence_research_cga{(' ' + email) if email else ''}"}

    for loc in locations:
        try:
            params = {"q": loc, "format": "json", "polygon_geojson": 1, "limit": 1}
            r = requests.get(NOMINATIM_URL, params=params, headers=headers, timeout=20)
            r.raise_for_status()
            results = r.json()
            if not results:
                print(f"⚠️ No results for {loc}")
                continue
            res = results[0]
            if 'geojson' in res and res['geojson']:
                pts = _coords_from_geojson(res['geojson'])
                if pts:
                    lats = [p[0] for p in pts]
                    lons = [p[1] for p in pts]
                    lat_min, lat_max = min(lats), max(lats)
                    lon_min, lon_max = min(lons), max(lons)
                    source = 'polygon'
                else:
                    source = 'none'
            elif 'boundingbox' in res:
                bb = res['boundingbox']
                lat_min, lat_max, lon_min, lon_max = map(float, bb)
                source = 'bbox'
            else:
                print(f"⚠️ No usable geometry for {loc}")
                continue

            if buffer_km and buffer_km > 0:
                mean_lat = (lat_min + lat_max) / 2.0
                deg_lat = buffer_km / 111.0
                deg_lon = buffer_km / (111.320 * math.cos(math.radians(mean_lat)))
                lat_min -= deg_lat
                lat_max += deg_lat
                lon_min -= deg_lon
                lon_max += deg_lon

            bboxes[loc] = {
                "lat_min": float(lat_min),
                "lat_max": float(lat_max),
                "lon_min": float(lon_min),
                "lon_max": float(lon_max),
                "source": source
            }
            print(f"✅ Successfully found bounding box for {loc} (source={source})")
        except Exception as e:
            print(f"Error fetching data for {loc}: {e}")

    with open(output_file, "w") as f:
        json.dump(bboxes, f, indent=4)
    print(f"\nSaved bounding boxes to {output_file}")
    return bboxes

# Run bounding box collection
BBOXES = collect_bounding_boxes(list(INCIDENTS.keys()), buffer_km=10, email="email@example.com")

✅ Successfully found bounding box for Uvalde, TX (source=polygon)
✅ Successfully found bounding box for Highland Park, IL (source=polygon)
✅ Successfully found bounding box for Monterey Park, CA (source=polygon)
✅ Successfully found bounding box for Nashville, TN (source=polygon)

Saved bounding boxes to /n/home10/jrochatrindade/gun-violence-twitter-forecasting/data/bboxes.json


## 2.1 Visualize Bounding Boxes on Map

In [15]:
# Visualize bounding boxes on an interactive map
try:
    import folium
    from folium import Rectangle
    # Ensure BBOXES exists and is not empty
    if not isinstance(BBOXES, dict) or len(BBOXES) == 0:
        print('No bounding boxes found in BBOXES. Run the bounding-box collection cell first.')
    else:
        # center map on mean of all bbox centers
        centers_lat = [(v['lat_min'] + v['lat_max']) / 2 for v in BBOXES.values()]
        centers_lon = [(v['lon_min'] + v['lon_max']) / 2 for v in BBOXES.values()]
        center = [sum(centers_lat) / len(centers_lat), sum(centers_lon) / len(centers_lon)]
        m = folium.Map(location=center, zoom_start=6)
        for city, bbox in BBOXES.items():
            bounds = [(bbox['lat_min'], bbox['lon_min']), (bbox['lat_max'], bbox['lon_max'])]
            Rectangle(bounds=bounds, color='red', weight=2, fill=True, fill_color='red', fill_opacity=0.2, popup=city).add_to(m)
            # add a marker at the bbox center for easier clicking
            folium.Marker(location=[(bbox['lat_min'] + bbox['lat_max'])/2, (bbox['lon_min'] + bbox['lon_max'])/2], popup=city).add_to(m)

        # Save map to HTML file
        html_output_path = os.path.join(OUTPUT_DIR, "bounding_boxes_map.html")
        m.save(html_output_path)
        print(f"✅ Map saved to {html_output_path}")
        
        display(m)
        
except ImportError:
    print('Folium is not installed. Install it with: pip install folium')
    print('\nBounding boxes:')
    for city, bbox in BBOXES.items():
        print(f"{city}: {bbox}")
except Exception as e:
    print('Error creating map:', e)
    print('\nBounding boxes:')
    for city, bbox in BBOXES.items():
        print(f"{city}: {bbox}")

✅ Map saved to /n/home10/jrochatrindade/gun-violence-twitter-forecasting/data/processed/bounding_boxes_map.html


In [8]:
# Print bbox dimensions for each city
print("Bounding box dimensions (width x height in degrees and approximate km):\n")
for city, bbox in BBOXES.items():
    lat_span = bbox['lat_max'] - bbox['lat_min']
    lon_span = bbox['lon_max'] - bbox['lon_min']
    
    # Approximate km (111 km per degree latitude, longitude varies by latitude)
    lat_km = lat_span * 111
    mean_lat = (bbox['lat_min'] + bbox['lat_max']) / 2
    lon_km = lon_span * 111.320 * abs(__import__('math').cos(__import__('math').radians(mean_lat)))
    
    print(f"{city}:")
    print(f"  Lat span: {lat_span:.4f}° ({lat_km:.1f} km)")
    print(f"  Lon span: {lon_span:.4f}° ({lon_km:.1f} km)")
    print(f"  Area: ~{lat_km * lon_km:.0f} km²\n")

Bounding box dimensions (width x height in degrees and approximate km):

Uvalde, TX:
  Lat span: 0.7220° (80.1 km)
  Lon span: 0.9070° (88.0 km)
  Area: ~7053 km²

Highland Park, IL:
  Lat span: 0.2502° (27.8 km)
  Lon span: 0.3357° (27.7 km)
  Area: ~769 km²

Monterey Park, CA:
  Lat span: 0.2247° (24.9 km)
  Lon span: 0.2934° (27.1 km)
  Area: ~675 km²

Nashville, TN:
  Lat span: 0.6179° (68.6 km)
  Lon span: 0.7618° (68.4 km)
  Area: ~4694 km²



## 3. Location and Date Filtering

In [9]:
# Precompute date ranges for each incident (t-2 to t+2 days)
INCIDENT_DATE_RANGES = {}
print("=== Configuring Spatio-Temporal Target Windows ===")
for city, date_str in INCIDENTS.items():
    incident_date = pd.to_datetime(date_str)
    start_date = incident_date - timedelta(days=2)
    end_date = incident_date + timedelta(days=2)
    
    # Safely extract bounding box pulled from previous step
    bbox_data = BBOXES.get(city)
    
    INCIDENT_DATE_RANGES[city] = {
        "start": start_date,
        "end": end_date,
        "bbox": bbox_data
    }
    
    print(f"{city:?<20} | Target: {incident_date.strftime('%Y-%m-%d')} | {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    if bbox_data:
        print(f"         Bounding Box -> Lat: [{bbox_data['lat_min']}, {bbox_data['lat_max']}] | Lon: [{bbox_data['lon_min']}, {bbox_data['lon_max']}]")
    else:
        print(f"         WARNING: Missing Bounding Box bounds for {city}!")
        
print("==================================================\n")

def process_file(file_path):
    """
    Reads a gzipped CSV of tweets from CGA and filters them by date and location.
    Assumes tab-separated values.
    """
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            df = pd.read_csv(f, sep='\t', on_bad_lines="skip", dtype=str)

        if 'date' not in df.columns or 'latitude' not in df.columns or 'longitude' not in df.columns:
            return pd.DataFrame()

        df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
        df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
        df['tweet_date'] = pd.to_datetime(df['date'], errors='coerce')
        df = df.dropna(subset=['tweet_date', 'latitude', 'longitude', 'text'])

        matched_tweets = []
        for city, conditions in INCIDENT_DATE_RANGES.items():
            start_date = conditions['start']
            end_date = conditions['end']
            bbox = conditions['bbox']
            
            if not bbox:
                continue

            date_mask = (df['tweet_date'] >= start_date) & (df['tweet_date'] <= end_date)
            lat_mask = (df['latitude'] >= bbox['lat_min']) & (df['latitude'] <= bbox['lat_max'])
            lon_mask = (df['longitude'] >= bbox['lon_min']) & (df['longitude'] <= bbox['lon_max'])
            
            city_df = df[date_mask & lat_mask & lon_mask].copy()
            if not city_df.empty:
                city_df['matched_incident'] = city
                matched_tweets.append(city_df)

        if matched_tweets:
            return pd.concat(matched_tweets, ignore_index=True)
        else:
            return pd.DataFrame()
    except Exception as e:
        return pd.DataFrame()

def filter_locations_dates():
    print("=== Starting Spatio-Temporal File Scraping ===")
    print(f"Scanning base source path: {RAW_DATA_DIR}")
    
    # Sequentially find files matching TARGET_YEARS ("2022", "2023")
    all_files = []
    for year in TARGET_YEARS:
        year_path = os.path.join(RAW_DATA_DIR, year, "**", "*.csv.gz")
        found_files = glob.glob(year_path, recursive=True)
        all_files.extend(found_files)
        print(f" Year {year}: Discovered {len(found_files)} compressed archives.")
        
    print(f" Cumulative payload target size: {len(all_files)} total files.")
    
    if not all_files:
        print(f" CRITICAL: No targeted files found in {RAW_DATA_DIR} for years {TARGET_YEARS}")
        return None

    NUM_CORES = min(20, os.cpu_count() or 1)
    print(f" Deploying Parallel Multiprocessing Architecture using {NUM_CORES} Core Workers...")
    print(" Beginning asynchronous file ingestion loop. Monitor progress tracking metrics below:\n")

    results = []
    
    # Using text-based tqdm inside a standard list constructor over imap_unordered
    with Pool(processes=NUM_CORES) as pool:
        # mininterval=5 forces the bar to update its text every 5 seconds instead of every iteration
        for df in tqdm(pool.imap_unordered(process_file, all_files), total=len(all_files), desc="Filtering Geotweets", mininterval=5):
            if not df.empty:
                results.append(df)

    print("\n Asynchronous ingestion finished. Commencing compilation phases...")
    print(f" Aggregating data chunks from {len(results)} non-empty sub-dataframes...")
    
    if results:
        final_df = pd.concat(results, ignore_index=True)
        out_path = os.path.join(OUTPUT_DIR, "location_date_filtered_tweets.csv.gz")
        
        print(f" Writing unified compressed structure matrix to target storage disk...")
        final_df.to_csv(out_path, index=False, compression="gzip", sep='\t')
        print(f" SUCCESS! Extracted and verified {len(final_df)} target tweets matching filters.")
        print(f"[ Output path reference: {out_path}")
        print("=== Execution Pipeline Safely Complete ===\n")
        return out_path
    else:
        print(" No tweets matched the geographic and temporal validation metrics.")
        print("========================================================================\n")
        return None

# Execute the pipeline
output_spatial_file = filter_locations_dates()

#if output_spatial_file:
# Trigger the keyword filtering phase
#    filter_locations_dates()
#    print("\nPROCESS COMPLETED")
#else:
#    print("\nPROCESS ERROR: No data generated")

print("\nPROCESS COMPLETED")

=== Configuring Spatio-Temporal Target Windows ===
Uvalde, TX?????????? | Target: 2022-05-24 | 2022-05-22 to 2022-05-26
         Bounding Box -> Lat: [28.99618250990991, 29.71817909009009] | Lon: [-100.21571832715628, -99.30871457284373]
Highland Park, IL??? | Target: 2022-07-04 | 2022-07-02 to 2022-07-06
         Bounding Box -> Lat: [42.06217450990991, 42.31239809009009] | Lon: [-87.97329966498407, -87.63761253501593]
Monterey Park, CA??? | Target: 2023-01-21 | 2023-01-19 to 2023-01-23
         Bounding Box -> Lat: [33.9370293099099, 34.161776890090096] | Lon: [-118.27796790226584, -117.98453489773415]
Nashville, TN??????? | Target: 2023-03-27 | 2023-03-25 to 2023-03-29
         Bounding Box -> Lat: [35.877694909909906, 36.49558609009009] | Lon: [-87.16606737540492, -86.40428662459507]

=== Starting Spatio-Temporal File Scraping ===
Scanning base source path: /n/holylabs/LABS/cga/Lab/data/geo-tweets/cga-sbg-tweets/
 Year 2022: Discovered 8504 compressed archives.
 Year 2023: Discover

Filtering Geotweets: 100%|██████████| 12823/12823 [20:39<00:00, 10.34it/s]



 Asynchronous ingestion finished. Commencing compilation phases...
 Aggregating data chunks from 301 non-empty sub-dataframes...
 Writing unified compressed structure matrix to target storage disk...
 SUCCESS! Extracted and verified 17282 target tweets matching filters.
[ Output path reference: /n/home10/jrochatrindade/gun-violence-twitter-forecasting/data/processed/location_date_filtered_tweets.csv.gz
=== Execution Pipeline Safely Complete ===


PROCESS COMPLETED


## 4. Keyword Filtering

In [10]:
def filter_by_keywords():
    print("=== Starting Advanced Textual Keyword and Hashtag Extraction ===")
    
    input_file = os.path.join(OUTPUT_DIR, "location_date_filtered_tweets.csv.gz")
    output_file = os.path.join(OUTPUT_DIR, "final_filtered_tweets.csv.gz")

    if not os.path.exists(input_file):
        print(f"ERROR: Spatio-temporal input file {input_file} not found.")
        print(" Please execute 'filter_locations_dates()' successfully before running this module.")
        return

    print(f" Loading spatio-temporally bounded dataset from: {input_file}")
    df = pd.read_csv(input_file, compression='gzip', sep='\t', dtype=str)
    
    if 'text' not in df.columns:
        print(" ERROR: 'text' data matrix vector column not found in data frame structures.")
        print(f"   Available columns in input dataset: {df.columns.tolist()}")
        return

    print(f"Dataset dimensions before text processing pipeline: {len(df)} total rows.")
    
    # Drop rows missing core text strings and explicitly reset the index layout
    df = df.dropna(subset=['text']).reset_index(drop=True)
    print(f" Dataset dimensions after removing null values from 'text' column: {len(df)} rows.")
    
    # -------------------------------------------------------------------------
    # ADVANCED HYBRID REGEX COMPILATION LAYER (WITH USER VOCABULARY)
    # -------------------------------------------------------------------------
    print(" Compiling hybrid non-capturing regex pattern utilizing wildcard extensions...")
    
    # 1. Base semantic radicals that expand morphologically (e.g., shoot -> shooter, shooting)
    wildcard_keywords = [
        "shoot", "gun", "firearm", "weapon", "kill", "injur", "fatal", "violenc", 
        "legislat", "restrict", "prevent", "protect", "attack"
    ]
    
    # 2. Process your explicit Hashtags and Keywords arrays into standardized tokens
    hashtag_words = [h.replace("#", "").lower() for h in HASHTAGS]
    raw_exact_keywords = list(set([k.lower() for k in KEYWORDS] + hashtag_words))
    
    # 3. Optimization: Exclude terms already naturally covered by wildcard expansion branches
    # This keeps the regex string matrix compact and highly performant on the cluster
    exact_keywords = []
    for kw in raw_exact_keywords:
        if not any(radical in kw for radical in wildcard_keywords):
            exact_keywords.append(re.escape(kw))
            
    # Escape the wildcard components and append the suffix token selector
    escaped_wildcards = [f"{re.escape(w)}\\w*" for w in wildcard_keywords]
    
    # 4. Construct final multi-branch alternation regex bound to exact word borders
    hybrid_pattern = r"\b(?:" + "|".join(escaped_wildcards + exact_keywords) + r")\b"
    compiled_regex = re.compile(hybrid_pattern, flags=re.IGNORECASE)
    
    print(f" -> Dynamic Vocabulary Summary: {len(wildcard_keywords)} wildcards, {len(exact_keywords)} optimized exact expressions.")
    # -------------------------------------------------------------------------
    
    print(" Scanning content arrays against compiled morphological regex vocabulary...")
    
    # Apply pattern match across the text series vector
    mask = df['text'].str.contains(compiled_regex, na=False, regex=True)
    final_df = df[mask].copy()
    
    print(f" Semantic parsing completed successfully.")
    print(f" Success! Retained {len(final_df)} target tweets matching vocabulary constraints.")
    
    if len(df) > 0:
        print(f" Reduction Ratio: {((len(df) - len(final_df)) / len(df) * 100):.2f}% of input strings filtered out.")
    
    # Basic data structure sanity check output for validation logs
    if not final_df.empty:
        print("\n--- Evaluation Snippet (Sample of Matches Texts) ---")
        sample_size = min(3, len(final_df))
        sample_tweets = final_df['text'].sample(sample_size, random_state=42).values
        for idx, text_snippet in enumerate(sample_tweets, 1):
            clean_snippet = str(text_snippet).replace('\n', ' ')[:110]
            print(f" Sample {idx}: {clean_snippet}...")
        print("----------------------------------------------------\n")

        print(f"Writing final structured output corpus matrix to disk...")
        final_df.to_csv(output_file, index=False, compression='gzip', sep='\t')
        
        # Physical disk validation check
        if os.path.exists(output_file):
            print(f" Saved final dataset to output path: {output_file}")
            print(f" Physical file verification size: {os.path.getsize(output_file)} bytes.")
        else:
            print(" ERROR: File serialization failed. Check cluster directory path permissions.")
    else:
        print(" PROCESS HALTED: Filtered matrix is completely empty. No data matched the keywords pattern.")
        
    print("=== Vocabulary Filtering Pipeline Complete ===\n")

filter = filter_by_keywords()
print("\nPROCESS COMPLETED")

=== Starting Advanced Textual Keyword and Hashtag Extraction ===
 Loading spatio-temporally bounded dataset from: /n/home10/jrochatrindade/gun-violence-twitter-forecasting/data/processed/location_date_filtered_tweets.csv.gz
Dataset dimensions before text processing pipeline: 17282 total rows.
 Dataset dimensions after removing null values from 'text' column: 17282 rows.
 Compiling hybrid non-capturing regex pattern utilizing wildcard extensions...
 -> Dynamic Vocabulary Summary: 13 wildcards, 65 optimized exact expressions.
 Scanning content arrays against compiled morphological regex vocabulary...
 Semantic parsing completed successfully.
 Success! Retained 1646 target tweets matching vocabulary constraints.
 Reduction Ratio: 90.48% of input strings filtered out.

--- Evaluation Snippet (Sample of Matches Texts) ---
 Sample 1: At 11:00 PM CDT, 2 SSE Smyrna [Rutherford Co, TN] PUBLIC reports TSTM WND DMG. AT SMYRNA HIGH SCHOOL, A LIGHT ...
 Sample 2: @RepMTG Okay so you ARE just a fuck

In [14]:
# Display sample of final filtered dataset
print("=" * 80)
print("FINAL FILTERED DATASET - SAMPLE (First 10 rows)")
print("=" * 80)

final_output_file = os.path.join(OUTPUT_DIR, "final_filtered_tweets.csv.gz")

if os.path.exists(final_output_file):
    df_sample = pd.read_csv(final_output_file, compression='gzip', sep='\t', nrows=10)
    
    # Display only tweet_text column
    text_col = 'tweet_text' if 'tweet_text' in df_sample.columns else 'text'
    if text_col in df_sample.columns:
        print(f"\nTotal columns: {len(df_sample.columns)}")
        print(f"Columns: {list(df_sample.columns)}\n")
        for idx, text in enumerate(df_sample[text_col], 1):
            print(f"\n--- Tweet {idx} ---")
            print(text)
    else:
        print(f"Column '{text_col}' not found")
    
    print(f"\n{'=' * 80}")
    print(f"Sample loaded successfully from {final_output_file}")
else:
    print(f" File not found: {final_output_file}")
    print("Run the keyword filtering cell first: filter_by_keywords()")

FINAL FILTERED DATASET - SAMPLE (First 10 rows)

Total columns: 27
Columns: ['message_id', 'date', 'text', 'tags', 'tweet_lang', 'source', 'place', 'geom', 'retweets', 'tweet_favorites', 'photo_url', 'quoted_status_id', 'user_id', 'user_name', 'user_location', 'followers', 'friends', 'user_favorites', 'status', 'user_lang', 'latitude', 'longitude', 'data_source', 'GPS', 'spatialerror', 'tweet_date', 'matched_incident']


--- Tweet 1 ---
🔥 2,000K+ #Google searches 4 Texas school shooting 2h old https://t.co/Zf2HX4G684 https://t.co/QvpzZs2Mks 
#Uvalde 🌅 6:42 🌆 20:29 CDT
#Covid
#UvaldeCounty 05/21/2022
New Cases 15DMA 1 Δ-46.7%
New Deaths 15DMA 0 Δ+100.0% https://t.co/29qT5K6AF4

--- Tweet 2 ---
Pictured: Salvador Ramos, 18 yr old of Uvalde, TX

He ran from local police after killing his grandmother.  The chase ended when he crashed his truck into a Arroyo.  He shot at police & ended up in a near by school, where he killed 14 students & 1 teacher. Police shot killed him. https://t.co/LnmD